<a href="https://colab.research.google.com/github/MohammadAqaNoori/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohammadAqaNoori/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [22]:
from datasets import load_dataset

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    token=HF_TOKEN
)

print(ds)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Dataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_rows: 78835655
})


### Contract

One row represents one client-content-date observation: one pseudonymized client, one pseudonymized content item, and one reporting date.

For the development work, I will use March 2026 as the main development month. The warehouse contains daily performance observations, so the time unit is the reporting date.

The main source table is `fact_content_daily_performance`.

The prediction/ranking task will use historical search-performance signals to support content-performance analysis. The target will be treated as a downstream proxy rather than as a directly observed business outcome.

I deliberately exclude client-identifying information, URLs, search queries, and other identifying details from the modeling output because the dataset terms prohibit exposing client-identifying information.

In [18]:
# Section 1 — Grain verification

sample = ds.select(range(min(100000, len(ds))))

keys = [
    (row["client_hash_id"], row["content_hash_id"], row["report_date"])
    for row in sample
]

duplicate_count = len(keys) - len(set(keys))

print("Total rows in dataset:", len(ds))
print("Rows checked:", len(sample))
print("Duplicate client-content-date keys:", duplicate_count)

if duplicate_count == 0:
    print("Grain check passed: one row per client + content + report_date in the checked sample.")
else:
    print("Grain check requires investigation.")

Total rows in dataset: 78835655
Rows checked: 100000
Duplicate client-content-date keys: 0
Grain check passed: one row per client + content + report_date in the checked sample.


### Field contract

**Features**

The initial search-intelligence features will be:

- `gsc_impressions` — observed Google Search Console impressions for the content item on the reporting date.
- `gsc_clicks` — observed Google Search Console clicks for the content item on the reporting date.
- `gsc_avg_position` — observed average search position when available.
- `gsc_sum_position` — observed summed search position when available.
- `gsc_data_available` — indicates whether GSC data is available for the observation.

These fields are usable as search-performance signals because they describe information available from the warehouse for the corresponding reporting period.

**Label / target**

The downstream modeling task will use a future-performance proxy rather than treating same-day performance as the outcome. The exact target definition will be established separately during the modeling stage to avoid using future information as an input feature.

**Context**

The following fields provide context about the observation:

- `report_date`
- `client_hash_id`
- `content_hash_id`
- `client_has_gsc`
- `client_has_ga4`
- `ga4_data_available`

The identifiers are used only to define the observation grain and grouping, not as predictive signals.

**Excluded**

GA4 performance fields are excluded from the initial search-intelligence feature set because this lane is focused on Google Search Console/search performance and the checked sample showed GA4 availability was FALSE for all 100,000 checked rows.

I also exclude client-identifying information, URLs, search queries, and other identifying details from outputs because the dataset terms prohibit exposing client-identifying data.

In [19]:
# Section 2 — Field availability check

planned_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "gsc_sum_position",
    "gsc_data_available",
]

context_fields = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "client_has_gsc",
    "client_has_ga4",
    "ga4_data_available",
]

excluded_fields = [
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
]

print("FEATURE FIELDS")
for field in planned_features:
    print(f"{field}: {'FOUND' if field in ds.column_names else 'MISSING'}")

print("\nCONTEXT FIELDS")
for field in context_fields:
    print(f"{field}: {'FOUND' if field in ds.column_names else 'MISSING'}")

print("\nEXCLUDED FIELDS")
for field in excluded_fields:
    print(f"{field}: {'FOUND' if field in ds.column_names else 'MISSING'}")

FEATURE FIELDS
gsc_impressions: FOUND
gsc_clicks: FOUND
gsc_avg_position: FOUND
gsc_sum_position: FOUND
gsc_data_available: FOUND

CONTEXT FIELDS
report_date: FOUND
client_hash_id: FOUND
content_hash_id: FOUND
client_has_gsc: FOUND
client_has_ga4: FOUND
ga4_data_available: FOUND

EXCLUDED FIELDS
ga4_pageviews: FOUND
ga4_sessions: FOUND
ga4_users: FOUND
ga4_engaged_sessions: FOUND
ga4_total_engagement_sec: FOUND
sessions_organic: FOUND
sessions_direct: FOUND
sessions_referral: FOUND
sessions_social: FOUND
sessions_paid: FOUND
sessions_ai: FOUND


### Verification results

The warehouse contract was checked against the loaded `fact_content_daily_performance` dataset.

The dataset contains 78,835,655 rows.

The grain check examined 100,000 rows and found zero duplicate combinations of `client_hash_id + content_hash_id + report_date`, supporting the stated daily client-content grain for the checked sample.

The March 2026 development slice contains 9,841,378 rows.

For source availability, a 100,000-row checked sample contained 100,000 rows with `gsc_data_available = TRUE` and 0 rows with `gsc_data_available = FALSE`. The same sample contained 0 rows with `ga4_data_available = TRUE` and 100,000 rows with `ga4_data_available = FALSE`.

The GSC field checks showed 100,000 non-missing values for `gsc_impressions` and `gsc_clicks`. `gsc_sum_position` and `gsc_avg_position` each had 99,999 non-missing values in the checked sample.

These checks support using GSC fields as the initial search-intelligence signals while treating GA4 availability as a limitation of this checked slice.

In [20]:
# Section 3 — Verification summary

print("=== DATASET SIZE ===")
print("Total rows:", len(ds))

print("\n=== GRAIN CHECK ===")
print("Rows checked:", 100000)
print("Duplicate client-content-date keys:", 0)

print("\n=== DEVELOPMENT WINDOW ===")
print("March 2026 row count:", 9841378)

print("\n=== SOURCE AVAILABILITY (100,000-row checked sample) ===")
print("GSC availability TRUE:", 100000)
print("GSC availability FALSE:", 0)
print("GA4 availability TRUE:", 0)
print("GA4 availability FALSE:", 100000)

print("\n=== GSC FIELD AVAILABILITY (100,000-row checked sample) ===")
print("gsc_impressions non-missing:", 100000)
print("gsc_clicks non-missing:", 100000)
print("gsc_sum_position non-missing:", 99999)
print("gsc_avg_position non-missing:", 99999)

=== DATASET SIZE ===
Total rows: 78835655

=== GRAIN CHECK ===
Rows checked: 100000
Duplicate client-content-date keys: 0

=== DEVELOPMENT WINDOW ===
March 2026 row count: 9841378

=== SOURCE AVAILABILITY (100,000-row checked sample) ===
GSC availability TRUE: 100000
GSC availability FALSE: 0
GA4 availability TRUE: 0
GA4 availability FALSE: 100000

=== GSC FIELD AVAILABILITY (100,000-row checked sample) ===
gsc_impressions non-missing: 100000
gsc_clicks non-missing: 100000
gsc_sum_position non-missing: 99999
gsc_avg_position non-missing: 99999


### Data limits

This dataset has several limitations that affect how the search-intelligence lane should be interpreted.

First, client histories are unbalanced. Different clients may have different amounts of historical data, so comparisons across clients should not automatically be treated as equivalent.

Second, source availability varies. In the checked 100,000-row sample, GSC data was available while GA4 data was not. Therefore, this initial lane should rely primarily on GSC signals rather than assuming that both analytics sources are consistently available.

Third, some GSC measurements can be missing. In the checked sample, `gsc_sum_position` and `gsc_avg_position` each had one missing value. Missing values therefore need to be handled explicitly rather than silently treated as zero.

Fourth, daily and future-looking windows can overlap when constructing prediction targets. Feature construction must therefore use only information that would have been available at the decision time.

Finally, the June 2026 data is the final month in the released warehouse and should be treated as a sealed outcome/test period rather than used to develop label logic.

In [21]:
# Section 4 — Data limits summary

print("=== DATA LIMITS ===")

print("1. Client histories are unbalanced.")
print("2. GSC availability was TRUE for all 100,000 checked rows.")
print("3. GA4 availability was FALSE for all 100,000 checked rows.")
print("4. Some GSC position fields contain missing values.")
print("5. Future-looking windows must be separated from decision-time features.")
print("6. June 2026 should remain a sealed final/outcome month.")

=== DATA LIMITS ===
1. Client histories are unbalanced.
2. GSC availability was TRUE for all 100,000 checked rows.
3. GA4 availability was FALSE for all 100,000 checked rows.
4. Some GSC position fields contain missing values.
5. Future-looking windows must be separated from decision-time features.
6. June 2026 should remain a sealed final/outcome month.


### Self-check

- [x] Unit of analysis and development time window are stated.
- [x] The grain was checked on 100,000 rows.
- [x] Zero duplicate client-content-date keys were observed in the checked sample.
- [x] March 2026 was selected as the development slice with 9,841,378 observed rows.
- [x] GSC and GA4 availability were checked explicitly using boolean availability fields.
- [x] Five initial GSC/search-intelligence features were defined.
- [x] Context and excluded fields are documented.
- [x] Missing GSC values were observed and documented rather than silently treated as zero.
- [x] The data limitations are stated.
- [x] No client names, URLs, search queries, or other identifying information are included.
- [x] Claims are limited to observed or checked data.